# 01 · Prepare CIC-IDS2017 Packet-Fields

Turns the 18 Packet-Fields files into API-shaped entries + labels with `nla_ml.adapters.cicids2017_packets`, tags each file with a split, and records per-file and per-attack-family stats.

**Out** (`/kaggle/working/cicids2017_prepared/`):
- `file_NN.parquet`: one per source file, in `packet_id` (capture) order. Columns: `file`, `split`, `packet_id`, the API entry fields (`src_ip` … `flags`), `flow_id`, `attack`, `is_attack`.
- `stats_file_NN.json`: per-file stats (lets a crashed run resume where it stopped).
- `manifest.json`: splits, held-out families, nla-ml commit, feature version/fingerprint, all stats.

This step drops only rows the API can't accept (counted per reason). **No deduplication or benign downsampling here**: those are modelling choices and belong in the training notebooks, which can make them differently per model. Test data must keep its real distribution so false-positive rates are honest.

**Needs:** internet on. Reads ~11 of 261 columns per file straight from Hugging Face, so memory stays at a few GB per file. Never read the whole file: each one is a single huge row group.

In [ ]:
!pip install -q "nla-ml[sklearn] @ git+https://github.com/HannanAFC/NetLogAnalyser.git@56e3e4ebc8c87b0cfbcdd726fde8a3e09254d832#subdirectory=backend/ml"

In [ ]:
import gc
import json
import time
from collections import Counter
from pathlib import Path

import nla_ml
import numpy as np
import pandas as pd
from nla_ml.adapters.cicids2017_packets import BENIGN_LABEL, DATASET, adapt_packet_fields, read_packet_fields
from nla_ml.features import FEATURE_FINGERPRINT, FEATURE_VERSION, featurize_frame

NLA_ML_COMMIT = "56e3e4ebc8c87b0cfbcdd726fde8a3e09254d832"   # keep equal to the pip install pin above
HF_ROOT       = "datasets/rdpahalavan/CIC-IDS2017/Packet-Fields"
LOCAL_DIR     = None                                          # or a folder of downloaded Packet_Fields_File_N.parquet
OUT           = Path( "/kaggle/working/cicids2017_prepared" )

print( f"nla-ml { nla_ml.__version__ } @ { NLA_ML_COMMIT[ :7 ] }, features v{ FEATURE_VERSION } ({ FEATURE_FINGERPRINT })" )

## Splits

Split **by file**: each file is a contiguous block of capture time, so train and test never share packets from the same time window. A pure day split doesn't work for CIC-IDS2017, because each day has different attacks, so every test attack would be unseen.

| Split | Files | Attacks in it |
|---|---|---|
| train | 1, 4, 5, 7, 9, 10, 12, 15, 18 | FTP/SSH-Patator, DoS slowloris / Slowhttptest / Hulk, Web Brute Force, DDoS |
| val | 2, 3 | 3.5 M benign + FTP-Patator: for choosing thresholds at a target false-positive rate |
| test | 6, 8, 11, 13, 14, 16, 17 | seen: FTP/SSH-Patator, DoS Hulk / slowloris, Web Brute Force, DDoS. Unseen: **PortScan**, Bot, GoldenEye, Heartbleed, XSS, SQLi, Infiltration |

- **Held out:** PortScan (files 16, 17) appears only in test, to measure detection of an attack family the models have never seen. Bot comes with it, because file 16 is its only file.
- **Single-file families:** some families exist in only one file, so they land in one split only. GoldenEye, Heartbleed (file 11), XSS, SQL Injection (13) and Infiltration (14) are test-only, so they are also unseen. Slowhttptest is train-only. "Seen" means *in train*: val families aren't trained on. Report test recall for seen and unseen families separately.
- **Val** is benign-heavy on purpose: it's for picking a threshold at a target false-positive rate. Tune hyperparameters with leave-one-file-out cross-validation over the train files.
- **Limitation:** neighbouring files can contain the same attack session (e.g. FTP-Patator in 5 and 6), so seen-family recall is optimistic. The unseen families give the honest number.

Change `SPLITS` / `HELD_OUT_FAMILIES` here; the checks below enforce them.

In [ ]:
SPLITS: dict[ int, str ] = {
    1: "train",  2: "val",    3: "val",    4: "train",  5: "train",  6: "test",
    7: "train",  8: "test",   9: "train", 10: "train", 11: "test",  12: "train",
   13: "test",  14: "test",  15: "train", 16: "test",  17: "test",  18: "train",
}
HELD_OUT_FAMILIES = [ "PortScan" ]   # must not appear in train or val

In [ ]:
def open_source( n: int ):
    if LOCAL_DIR:
        return open( Path( LOCAL_DIR ) / f"Packet_Fields_File_{ n }.parquet", "rb" )
    from huggingface_hub import HfFileSystem
    return HfFileSystem( ).open( f"{ HF_ROOT }/Packet_Fields_File_{ n }.parquet", "rb", block_size=8 * 1024**2 )


def overlap_stats( features: pd.DataFrame, labels: pd.DataFrame ) -> dict:
    """How many attack packets have a feature row that also occurs as benign in the same file,
    i.e. that per-packet features can't tell apart from normal traffic."""
    row_hash = pd.util.hash_pandas_object( features, index=False ).to_numpy( )
    is_attack = labels[ "is_attack" ].to_numpy( ) == 1
    also_benign = np.isin( row_hash, np.unique( row_hash[ ~is_attack ] ) ) & is_attack
    attacks = labels[ "attack" ].to_numpy( )
    per_family = {
        family: { "packets": int( ( attacks == family ).sum( ) ), "also_benign": int( also_benign[ attacks == family ].sum( ) ) }
        for family in sorted( set( attacks[ is_attack ] ) )
    }
    return {
        "distinct_feature_rows": int( len( np.unique( row_hash ) ) ),
        "attack_packets":        int( is_attack.sum( ) ),
        "attack_also_benign":    int( also_benign.sum( ) ),
        "per_family":            per_family,
    }


def prepare_file( n: int, split: str ) -> dict:
    t0 = time.perf_counter( )
    with open_source( n ) as f:
        raw = read_packet_fields( f )
    adapted = adapt_packet_fields( raw )
    del raw

    features = featurize_frame( adapted.entries )   # only for the overlap stats; training notebooks re-featurize
    overlap  = overlap_stats( features, adapted.labels )
    del features

    frame = adapted.entries.join( adapted.labels ).reset_index( )
    frame.insert( 0, "file", np.int8( n ) )
    frame.insert( 1, "split", split )
    out_path = OUT / f"file_{ n:02d}.parquet"
    frame.to_parquet( out_path, compression="zstd", index=False )

    stats = {
        "file": n, "split": split,
        "rows_in": adapted.total, "kept": adapted.kept, "dropped": adapted.dropped,
        "attacks": { k: int( v ) for k, v in adapted.labels[ "attack" ].value_counts( ).items( ) if k != BENIGN_LABEL },
        "benign": int( ( adapted.labels[ "is_attack" ] == 0 ).sum( ) ),
        **overlap,
        "seconds": round( time.perf_counter( ) - t0, 1 ),
        "out_mb": round( out_path.stat( ).st_size / 1e6, 1 ),
    }
    ( OUT / f"stats_file_{ n:02d}.json" ).write_text( json.dumps( stats, indent=2 ) )
    del adapted, frame
    gc.collect( )
    return stats

## Run

Processes one file at a time, roughly 30 s to a few minutes each (mostly download). Files that already have a `stats_file_NN.json` are skipped, so if the session dies, re-run this cell.

In [ ]:
OUT.mkdir( parents=True, exist_ok=True )
stats: dict[ int, dict ] = { }
for n, split in SPLITS.items( ):
    done = OUT / f"stats_file_{ n:02d}.json"
    if done.exists( ) and ( OUT / f"file_{ n:02d}.parquet" ).exists( ):
        stats[ n ] = json.loads( done.read_text( ) )
        print( f"file { n:2d} ({ split }): already done" )
        continue
    stats[ n ] = prepare_file( n, split )
    s = stats[ n ]
    print( f"file { n:2d} ({ split:5}): { s[ 'kept' ]:>10,} kept, dropped { s[ 'dropped' ] or '{}' }, "
           f"attacks { s[ 'attacks' ] or '{}' } [{ s[ 'seconds' ] }s, { s[ 'out_mb' ] } MB]" )

## Checks

In [ ]:
by_split = { split: Counter( ) for split in ( "train", "val", "test" ) }
for s in stats.values( ):
    by_split[ s[ "split" ] ][ BENIGN_LABEL ] += s[ "benign" ]
    by_split[ s[ "split" ] ].update( s[ "attacks" ] )

split_table = pd.DataFrame( by_split ).fillna( 0 ).astype( "int64" )
split_table.loc[ "TOTAL" ] = split_table.sum( )
display( split_table )

families = { split: { f for f in c if f != BENIGN_LABEL } for split, c in by_split.items( ) }
seen   = families[ "train" ]            # what the models learn from
tested = families[ "test" ]
leaked = sorted( set( HELD_OUT_FAMILIES ) & ( families[ "train" ] | families[ "val" ] ) )
assert not leaked, f"held-out families appear in train/val: { leaked }"
assert set( HELD_OUT_FAMILIES ) <= tested, "every held-out family must appear in test"
assert set( SPLITS ) == set( stats ), "not every file was processed"

print( "test, seen in train:", sorted( tested & seen ) )
print( "test, unseen:       ", sorted( tested - seen ), "(deliberately held out:", HELD_OUT_FAMILIES, ")" )
print( "val:                ", sorted( families[ "val" ] ) )
print( "train only:         ", sorted( seen - tested ) )

In [ ]:
# Per family: share of attack packets whose feature row also occurs as benign in the same file.
# High share = per-packet features can't separate that attack (a case for the Phase 7 window features).
rows = { }
for s in stats.values( ):
    for family, c in s[ "per_family" ].items( ):
        r = rows.setdefault( family, { "packets": 0, "also_benign": 0, "files": [ ] } )
        r[ "packets" ] += c[ "packets" ]
        r[ "also_benign" ] += c[ "also_benign" ]
        r[ "files" ].append( s[ "file" ] )
separability = pd.DataFrame( rows ).T
separability[ "share_also_benign" ] = ( separability[ "also_benign" ] / separability[ "packets" ] ).astype( float ).round( 3 )
display( separability.sort_values( "share_also_benign", ascending=False ) )

per_file = pd.DataFrame( [
    { k: s[ k ] for k in ( "file", "split", "rows_in", "kept", "benign", "attack_packets", "distinct_feature_rows", "seconds", "out_mb" ) }
    | { "dropped": sum( s[ "dropped" ].values( ) ) }
    for s in stats.values( )
] ).set_index( "file" )
display( per_file )

In [ ]:
manifest = {
    "dataset":             DATASET,
    "subset":              "Packet-Fields",
    "source":              HF_ROOT,
    "nla_ml_version":      nla_ml.__version__,
    "nla_ml_commit":       NLA_ML_COMMIT,
    "feature_version":     FEATURE_VERSION,
    "feature_fingerprint": FEATURE_FINGERPRINT,
    "splits":              { str( n ): split for n, split in SPLITS.items( ) },
    "held_out_families":   HELD_OUT_FAMILIES,
    "families":            { "train": sorted( seen ), "val": sorted( families[ "val" ] ), "test_seen": sorted( tested & seen ), "test_unseen": sorted( tested - seen ) },
    "split_label_counts":  { split: dict( c ) for split, c in by_split.items( ) },
    "files":               { str( n ): s for n, s in stats.items( ) },
    "notes": [
        "No deduplication or benign downsampling: do that in training notebooks, never on test.",
        "captured_at is not stored: the subset has no timestamps; packet_id gives capture order.",
    ],
}
( OUT / "manifest.json" ).write_text( json.dumps( manifest, indent=2 ) )
print( f"wrote { OUT / 'manifest.json' }; total output { sum( p.stat( ).st_size for p in OUT.iterdir( ) ) / 1e9:.2f} GB" )

## Publish

Save a version of this notebook (**Save Version → Save & Run All**). Then, from the version's **Output** tab, choose **New Dataset** (first time) or **New Version** of the existing dataset, e.g. `nla-cicids2017-prepared`. Training notebooks add that dataset as an input, read `manifest.json` first, and load files with `nla_ml.adapters.cicids2017_packets.read_prepared`. Plain `pd.read_parquet` turns missing `flags` into NaN under pandas 3: that's fine for features and heuristics, but the API rejects it when replaying entries.

Record the dataset version you publish. It goes into each model's metadata as `dataset_version`.